# LC 739 — Daily Temperatures
**Difficulty:** Medium | **Category:** Stack | **Pattern:** Monotonic Stack

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Maintain a stack of indices whose
temperatures are still waiting for a warmer day. When a new
temperature is warmer than the top index, pop it and record
the gap. The stack stays in decreasing temperature order
(monotonic decreasing stack), giving O(n) overall.
</div>

## Official Problem Statement

Given an array of integers `temperatures` represents the daily
temperatures, return an array `answer` such that `answer[i]` is
the number of days you have to wait after the `i`-th day to get
a warmer temperature.

If there is no future day with a warmer temperature, keep
`answer[i] == 0`.

**Constraints:**
- `1 <= temperatures.length <= 10^5`
- `30 <= temperatures[i] <= 100`

## What This Is Actually Asking

For each day, find how many days in the future you have to look
before the temperature is strictly higher than today's.

If no warmer day ever comes, the answer for that day is 0.

A brute-force scan of every future day for every day would be
O(n²) — too slow for 100,000 days.

The key insight: a day "waiting" for warmth can be stored and
resolved in one pass using a stack, achieving O(n).

## Walk Through an Example by Hand

Input:  `temperatures = [73, 74, 75, 71, 69, 72, 76, 73]`
Output: `[1,  1,  4,  2,  1,  1,  0,  0]`

```
i=0 t=73  stack=[]       push 0     stack=[0]
i=1 t=74  74>temps[0]=73 pop 0, ans[0]=1-0=1
                          push 1     stack=[1]
i=2 t=75  75>temps[1]=74 pop 1, ans[1]=2-1=1
                          push 2     stack=[2]
i=3 t=71  71<temps[2]=75 push 3     stack=[2,3]
i=4 t=69  69<temps[3]=71 push 4     stack=[2,3,4]
i=5 t=72  72>temps[4]=69 pop 4, ans[4]=5-4=1
          72>temps[3]=71 pop 3, ans[3]=5-3=2
          72<temps[2]=75 stop
                          push 5     stack=[2,5]
i=6 t=76  76>temps[5]=72 pop 5, ans[5]=6-5=1
          76>temps[2]=75 pop 2, ans[2]=6-2=4
                          push 6     stack=[6]
i=7 t=73  73<temps[6]=76 push 7     stack=[6,7]

Remaining stack [6,7]: no warmer day → ans stays 0
```

## The Picture

The stack always holds indices of temperatures waiting for a
warmer day, in decreasing temperature order (top = coldest
unresolved day).

```
temps:  73  74  75  71  69  72  76  73
index:   0   1   2   3   4   5   6   7

After i=4 (t=69) — 3 days unresolved:
  top → | idx=4, t=69 |   ← coldest, most recent
         | idx=3, t=71 |
  bot → | idx=2, t=75 |   ← warmest waiting

At i=5 (t=72):
  72 > 69 → pop 4, ans[4] = 5-4 = 1
  72 > 71 → pop 3, ans[3] = 5-3 = 2
  72 < 75 → stop popping
  push 5

  top → | idx=5, t=72 |
  bot → | idx=2, t=75 |

Key property: temperatures in stack are always DECREASING
from bottom to top (monotonic decreasing stack of indices).
```

## When To Use This Pattern

- When you need the **next greater element** to the right for
  each position, think **monotonic decreasing stack**.
- When items are "waiting" for a future event that resolves
  them in LIFO order, think **stack**.
- When a brute-force double loop O(n²) seems obvious but too
  slow, think **monotonic stack** for O(n).
- When the answer involves a **distance or gap** to a future
  index, store **indices** (not values) in the stack.
- When you see "next larger", "next smaller", "days until",
  think **monotonic stack**.

## The Approach

Initialize a result array of zeros and an empty stack that will
hold indices of unresolved days.

For each index i, while the stack is non-empty and the current
temperature is warmer than the temperature at the top index,
pop that index and record the gap (i minus popped index) as its
answer.

After resolving all colder days, push the current index onto
the stack. Any indices still in the stack at the end have no
warmer future day and their answer remains 0.

The stack always stays in monotonically decreasing temperature
order, ensuring each index is pushed and popped at most once.

In [12]:
from typing import List  # type hint for temperature list

In [13]:
def test_harness(func):
    """
    Runs test cases for LC 739 Daily Temperatures.
    Each case: (temperatures_list, expected_result_list)
    """
    cases = [
        # (temps,                        expected)
        ([73,74,75,71,69,72,76,73],  [1,1,4,2,1,1,0,0]),
        ([30,40,50,60],               [1,1,1,0]),
        ([30,60,90],                  [1,1,0]),
        ([90,60,30],                  [0,0,0]),   # decreasing
        ([70,70,70],                  [0,0,0]),   # all same
        ([55],                        [0]),        # single day
        ([50,60,50,60,50],           [1,0,1,0,0]),
        ([34,80,34,80],              [1,0,1,0]),
    ]
    passed = 0
    for temps, expected in cases:
        result = func(temps)
        status = "PASSED" if result == expected else "FAILED"
        if status == "FAILED":
            print(
                f"  {status} | temps={temps}\n"
                f"           expected={expected}\n"
                f"           got     ={result}"
            )
        else:
            passed += 1
    total = len(cases)
    print(f"\nResult: {passed}/{total} tests passed.")

In [14]:
def daily_temperatures(temps: List[int]) -> List[int]:
    """
    Return wait-days array for next warmer temperature.

    Strategy:
      - result[i] = 0 by default (no warmer day found).
      - stack holds indices of unresolved (waiting) days.
      - For each i: while temps[stack[-1]] < temps[i]:
          popped = stack.pop(); result[popped] = i - popped
      - Push i onto stack.
      - Stack keeps decreasing temp order (monotonic dec).

    Args:
        temperatures: list of daily temperatures
    Returns:
        list of ints: days to wait for warmer temp
    """
    # Initialize a zero list for every day.
    # Initialize an empty stack to house indices of monotonic decreasing temperatures.
    # For a temperature warmer than the stack top, evict the index and populate the result.
    # Example: daily_temperatures([90, 60, 30]) -> [0, 0, 0] (a decreasing monotonic stack).
    
    stack, res = [], [0] * len(temps)
    for i, temp in enumerate(temps):
        while stack and temp > temps[stack[-1]]:
            idx = stack.pop()
            res[idx] = i - idx
        stack.append(i)
    return res


# --- Debug prints (remove before submitting) ---
"""
[1, 1, 4, 2, 1, 1, 0, 0]
[1, 1, 1, 0]
[0, 0, 0]
[0, 0, 0]
[0]

Result: 8/8 tests passed.
"""
# Main example — expect [1,1,4,2,1,1,0,0]
print(daily_temperatures([73,74,75,71,69,72,76,73]))

# Strictly increasing — expect [1,1,1,0]
print(daily_temperatures([30,40,50,60]))

# Strictly decreasing — expect [0,0,0]
print(daily_temperatures([90,60,30]))

# All same — expect [0,0,0]
print(daily_temperatures([70,70,70]))

# Single element — expect [0]
print(daily_temperatures([55]))
test_harness(daily_temperatures)

[1, 1, 4, 2, 1, 1, 0, 0]
[1, 1, 1, 0]
[0, 0, 0]
[0, 0, 0]
[0]

Result: 8/8 tests passed.


In [ ]:
# Uncomment and run when solution is ready
# test_harness(daily_temperatures)

## Complexity

| Approach             | Time   | Space  |
|----------------------|--------|--------|
| Brute force (nested loop) | O(n²) | O(1) |
| Optimal (monotonic stack) | O(n)  | O(n)  |

- **Time O(n):** each index is pushed once and popped at most
  once, so total operations are bounded by 2n.
- **Space O(n):** the stack can hold at most n indices in the
  worst case (monotonically decreasing temperatures).

## Real World Connection

In AWS CloudWatch, anomaly detection on metric streams uses a
monotonic-stack approach to find the next time a metric exceeds
a threshold — the same "next greater element" pattern as this
problem.

At Citi, financial alert systems scan trade price series to find
the next time a stock price rises above the current level for
stop-loss triggers. A monotonic stack processes the entire day's
feed in O(n) rather than re-scanning for each event.

In data engineering, streaming pipeline processors use
monotonic stacks to compute session windows — finding when
activity rises above idle thresholds across event logs without
expensive repeated scans of historical data.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra